In [ ]:
import re
from typing import Dict, List, Optional
from collections import Counter, defaultdict
import json as _json

In [ ]:
# ── PARTE 1: Parsers de Logs ─────────────────────────────────────────────

PATRON_HTTP = re.compile(r'''
    ^(?P<ip>\d{1,3}(?:\.\d{1,3}){3})   # Direccion IP del cliente
    \s+-\s+-\s+                          # Separadores
    \[(?P<timestamp>[^\]]+)\]            # Timestamp entre corchetes
    \s+"(?P<method>\w+)                  # Metodo HTTP
    \s+(?P<path>[^\s"]+)                 # Ruta solicitada
    \s+[^"]+?"                           # Protocolo
    \s+(?P<status>\d{3})                 # Codigo de estado
    \s+(?P<bytes>\d+)                    # Bytes transferidos
    \s+"(?P<referer>[^"]*)"              # Referer
    \s+"(?P<user_agent>[^"]*)"           # User-Agent
''', re.VERBOSE)

def parse_http_log(linea: str) -> Optional[Dict]:
    m = PATRON_HTTP.match(linea.strip())
    if not m:
        return None
    return {
        'ip':         m.group('ip'),
        'timestamp':  m.group('timestamp'),
        'method':     m.group('method'),
        'path':       m.group('path'),
        'status':     int(m.group('status')),
        'bytes':      int(m.group('bytes')),
        'referer':    m.group('referer'),
        'user_agent': m.group('user_agent'),
    }

In [ ]:
PATRON_ERROR = re.compile(r'''
    ^\[(?P<timestamp>\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\]  # Timestamp
    \s+(?P<level>DEBUG|INFO|WARNING|ERROR|CRITICAL)              # Nivel
    \s+(?P<module>[\w.]+)                                        # Modulo
    \s+-\s+(?P<error_type>\w+):\s+(?P<message>.+)$              # Tipo y mensaje
''', re.VERBOSE)

def parse_error_log(linea: str) -> Optional[Dict]:
    m = PATRON_ERROR.match(linea.strip())
    if not m:
        return None
    return {
        'timestamp':  m.group('timestamp'),
        'level':      m.group('level'),
        'module':     m.group('module'),
        'error_type': m.group('error_type'),
        'message':    m.group('message'),
    }

In [ ]:
# Usa lookbehind: extrae el valor inmediatamente despues de cada '='
PATRON_AUTH = re.compile(r'''
    ^\[AUTH\]\s+
    (?P<timestamp>\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})
    \s+\|\s+user=(?P<user>[^\s|]+)
    \s+\|\s+action=(?P<action>[^\s|]+)
    \s+\|\s+status=(?P<status>[^\s|]+)
    \s+\|\s+ip=(?P<ip>[^\s|]+)
    (?:\s+\|\s+(?P<extra_key>session|attempts)=(?P<extra_val>[^\s|]+))?
''', re.VERBOSE)

def parse_auth_log(linea: str) -> Optional[Dict]:
    m = PATRON_AUTH.match(linea.strip())
    if not m:
        return None
    extra = {}
    if m.group('extra_key'):
        val = m.group('extra_val')
        extra[m.group('extra_key')] = int(val) if m.group('extra_key') == 'attempts' else val
    return {
        'timestamp': m.group('timestamp'),
        'user':      m.group('user'),
        'action':    m.group('action'),
        'status':    m.group('status'),
        'ip':        m.group('ip'),
        'extra':     extra,
    }

In [ ]:
PATRON_DB = re.compile(r'''
    ^\[DB-(?P<timestamp>\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\]
    \s+(?P<query_type>SLOW_QUERY|QUERY)
    (?:
        \s+executed\s+in\s+(?P<time_normal>[\d.]+)s
        |
        \s+\((?P<time_slow>[\d.]+)s\)
    )
    :\s+(?P<query>.+)$
''', re.VERBOSE)

def parse_db_log(linea: str) -> Optional[Dict]:
    m = PATRON_DB.match(linea.strip())
    if not m:
        return None
    t = m.group('time_normal') or m.group('time_slow')
    return {
        'timestamp':      m.group('timestamp'),
        'query_type':     m.group('query_type'),
        'execution_time': float(t),
        'query':          m.group('query'),
    }

In [ ]:
# ── PARTE 2: Analizador de Seguridad ────────────────────────────────────

def detectar_ataques_fuerza_bruta(logs_auth: List[Dict]) -> List[Dict]:
    fallos: Dict[str, int] = defaultdict(int)
    for log in logs_auth:
        if log.get('status') == 'FAILED':
            fallos[log['ip']] += 1
    return [{'ip': ip, 'intentos': cnt} for ip, cnt in fallos.items() if cnt > 3]


PATRONES_SQL_INJECTION = [
   r"(?i)\bOR\b\s+['\"]?\d+['\"]?\s*=\s*['\"]?\d+",
    r"(?i)\bUNION\b.*\bSELECT\b",
    r"--",
    r"(?i)\bDROP\b\s+\bTABLE\b",
    r"(?i)\bDELETE\b\s+\bFROM\b.*\bWHERE\b\s+1\s*=\s*1",
]
_sql_re = [re.compile(p) for p in PATRONES_SQL_INJECTION]

def detectar_sql_injection(logs_db: List[Dict]) -> List[Dict]:
    sospechosos = []
    for log in logs_db:
        query = log.get('query', '')
        for patron in _sql_re:
            if patron.search(query):
                sospechosos.append(log)
                break
    return sospechosos


def detectar_path_traversal(logs_http: List[Dict]) -> List[Dict]:
    patron = re.compile(r'(?:\.\.[/\\]|%2e%2e(?:%2f|/))', re.IGNORECASE)
    return [log for log in logs_http if patron.search(log.get('path', ''))]


def detectar_errores_criticos(logs_error: List[Dict]) -> List[Dict]:
    criticos = [l for l in logs_error if l.get('level') in ('ERROR', 'CRITICAL')]
    return sorted(criticos, key=lambda x: x['timestamp'])

In [ ]:
# ── PARTE 3: Generador de Reportes ──────────────────────────────────────

def clasificar_linea(linea: str) -> str:
    l = linea.strip()
    if re.match(r'^\d{1,3}(?:\.\d{1,3}){3}', l): return 'http'
    if l.startswith('[AUTH]'):                     return 'auth'
    if l.startswith('[DB-'):                       return 'db'
    if re.match(r'^\[\d{4}', l):                return 'error'
    return 'desconocido'


def generar_reporte(logs: str) -> Dict:
    lineas = [l for l in logs.strip().splitlines() if l.strip()]

    tipos: Dict[str, List[str]] = {'http': [], 'error': [], 'auth': [], 'db': [], 'desconocido': []}
    for linea in lineas:
        tipos[clasificar_linea(linea)].append(linea)

    http_logs  = [r for r in (parse_http_log(l)  for l in tipos['http'])  if r]
    error_logs = [r for r in (parse_error_log(l) for l in tipos['error']) if r]
    auth_logs  = [r for r in (parse_auth_log(l)  for l in tipos['auth'])  if r]
    db_logs    = [r for r in (parse_db_log(l)    for l in tipos['db'])    if r]

    por_status = {'2xx': 0, '3xx': 0, '4xx': 0, '5xx': 0}
    for h in http_logs:
        s = h['status']
        if   200 <= s < 300: por_status['2xx'] += 1
        elif 300 <= s < 400: por_status['3xx'] += 1
        elif 400 <= s < 500: por_status['4xx'] += 1
        elif 500 <= s < 600: por_status['5xx'] += 1

    tiempos = [d['execution_time'] for d in db_logs]

    return {
        'resumen': {
            'total_lineas': len(lineas),
            'por_tipo': {k: len(v) for k, v in tipos.items() if k != 'desconocido'}
        },
        'http': {
            'total_requests': len(http_logs),
            'por_status':     por_status,
            'top_rutas':      Counter(h['path'] for h in http_logs).most_common(5),
            'top_ips':        Counter(h['ip']   for h in http_logs).most_common(5),
        },
        'errores': {
            'total':      len(error_logs),
            'por_nivel':  dict(Counter(e['level']  for e in error_logs)),
            'por_modulo': dict(Counter(e['module'] for e in error_logs)),
        },
        'seguridad': {
            'alertas_fuerza_bruta':   detectar_ataques_fuerza_bruta(auth_logs),
            'alertas_sql_injection':  detectar_sql_injection(db_logs),
            'alertas_path_traversal': detectar_path_traversal(http_logs),
        },
        'rendimiento': {
            'queries_lentos':          [d for d in db_logs if d['query_type'] == 'SLOW_QUERY'],
            'tiempo_promedio_queries': sum(tiempos) / len(tiempos) if tiempos else 0.0,
        }
    }

In [ ]:
def mostrar_reporte(reporte: Dict) -> None:
    print('=' * 70)
    print('                    REPORTE DE ANALISIS DE LOGS')
    print('=' * 70)

    print('\n Resumen GENERAL')
    print('-' * 40)
    print(f"Total de lineas procesadas: {reporte['resumen']['total_lineas']}")
    print('Por tipo:')
    for tipo, count in reporte['resumen']['por_tipo'].items():
        print(f'  - {tipo.upper()}: {count}')

    print('\n Logs HTTP')
    print('-' * 40)
    print(f"Total requests: {reporte['http']['total_requests']}")
    print('Por codigo de estado:')
    for status, count in reporte['http']['por_status'].items():
        print(f'  - {status}: {count}')
    print('Top 5 rutas mas solicitadas:')
    for ruta, count in reporte['http'].get('top_rutas', [])[:5]:
        print(f'  - {ruta}: {count} requests')

    print('\n Errores')
    print('-' * 40)
    print(f"Total errores: {reporte['errores']['total']}")
    print('Por nivel:')
    for nivel, count in reporte['errores']['por_nivel'].items():
        print(f'  - {nivel}: {count}')

    print('\n Alertas de Seguridad')
    print('-' * 40)
    fb = reporte['seguridad'].get('alertas_fuerza_bruta', [])
    if fb:
        print(f'Posibles ataques de fuerza bruta: {len(fb)}')
        for a in fb:
            print(f"  IP: {a['ip']} - {a['intentos']} intentos fallidos")
    sql = reporte['seguridad'].get('alertas_sql_injection', [])
    if sql:
        print(f'Posibles SQL Injection: {len(sql)}')
        for a in sql[:3]:
            print(f"  Query: {a['query'][:60]}...")
    pt = reporte['seguridad'].get('alertas_path_traversal', [])
    if pt:
        print(f'Posibles Path Traversal: {len(pt)}')
        for a in pt[:3]:
            print(f"  Ruta: {a['path']}")

    print('\n Rendimiento')
    print('-' * 40)
    print(f"Queries lentos: {len(reporte['rendimiento'].get('queries_lentos', []))}")
    print(f"Tiempo promedio: {reporte['rendimiento']['tiempo_promedio_queries']:.3f}s")
    print('\n' + '=' * 70)

In [ ]:
LOGS_PRUEBA = """
192.168.1.100 - - [15/Mar/2024:10:23:45 -0600] "GET /api/users HTTP/1.1" 200 1234 "https://ejemplo.com" "Mozilla/5.0 (Windows NT 10.0)"
192.168.1.101 - - [15/Mar/2024:10:23:46 -0600] "POST /api/login HTTP/1.1" 200 89 "-" "curl/7.68.0"
192.168.1.102 - - [15/Mar/2024:10:23:47 -0600] "GET /admin/../../../etc/passwd HTTP/1.1" 403 0 "-" "sqlmap/1.0"
[2024-03-15 10:24:00] INFO app.startup - Application started successfully on port 8080
[2024-03-15 10:25:12] ERROR app.database - DatabaseConnectionError: Connection refused to host db.server.com:5432
[2024-03-15 10:25:15] WARNING app.cache - CacheWarning: Redis connection timeout, using fallback
[2024-03-15 10:26:00] ERROR app.auth - AuthenticationError: Invalid token for user admin@empresa.com
[AUTH] 2024-03-15 10:30:00 | user=admin@empresa.com | action=LOGIN | status=SUCCESS | ip=10.0.0.5 | session=abc123xyz
[AUTH] 2024-03-15 10:31:00 | user=hacker@mail.com | action=LOGIN | status=FAILED | ip=192.168.1.50 | attempts=1
[AUTH] 2024-03-15 10:31:30 | user=hacker@mail.com | action=LOGIN | status=FAILED | ip=192.168.1.50 | attempts=2
[AUTH] 2024-03-15 10:32:00 | user=hacker@mail.com | action=LOGIN | status=FAILED | ip=192.168.1.50 | attempts=3
[AUTH] 2024-03-15 10:32:30 | user=hacker@mail.com | action=LOGIN | status=FAILED | ip=192.168.1.50 | attempts=4
[AUTH] 2024-03-15 10:33:00 | user=otro@empresa.com | action=LOGOUT | status=SUCCESS | ip=10.0.0.10 | session=def456uvw
[DB-2024-03-15 10:35:22] QUERY executed in 0.045s: SELECT * FROM users WHERE email = 'admin@empresa.com'
[DB-2024-03-15 10:35:25] QUERY executed in 0.012s: SELECT id, name FROM products WHERE active = 1
[DB-2024-03-15 10:36:00] SLOW_QUERY (2.5s): SELECT * FROM orders o JOIN products p ON o.product_id = p.id JOIN users u ON o.user_id = u.id
[DB-2024-03-15 10:37:00] QUERY executed in 0.001s: SELECT * FROM users WHERE username = 'admin' OR 1=1--'
[DB-2024-03-15 10:38:00] QUERY executed in 0.002s: SELECT * FROM users UNION SELECT * FROM passwords
192.168.1.200 - - [15/Mar/2024:10:40:00 -0600] "GET /products?id=1 HTTP/1.1" 200 5678 "https://tienda.com" "Mozilla/5.0"
192.168.1.200 - - [15/Mar/2024:10:40:05 -0600] "GET /products?id=2 HTTP/1.1" 200 4321 "https://tienda.com" "Mozilla/5.0"
192.168.1.201 - - [15/Mar/2024:10:41:00 -0600] "GET /api/users HTTP/1.1" 401 123 "-" "PostmanRuntime/7.26.8"
192.168.1.201 - - [15/Mar/2024:10:41:05 -0600] "GET /api/users HTTP/1.1" 500 0 "-" "PostmanRuntime/7.26.8"
[2024-03-15 10:42:00] ERROR app.api - NullPointerException: Cannot read property 'id' of undefined
[DB-2024-03-15 10:45:00] SLOW_QUERY (5.2s): SELECT COUNT(*) FROM logs WHERE date > '2024-01-01'
""".strip()

In [ ]:
print('PRUEBA DE PARSERS')
print('=' * 50)

linea_http = '192.168.1.100 - - [15/Mar/2024:10:23:45 -0600] "GET /api/users HTTP/1.1" 200 1234 "https://ejemplo.com" "Mozilla/5.0"'
print('\n-- Parser HTTP --')
print(f'Entrada: {linea_http[:60]}...')
print(f'Resultado: {parse_http_log(linea_http)}')

linea_error = '[2024-03-15 10:25:12] ERROR app.database - DatabaseConnectionError: Connection refused'
print('\n-- Parser Error --')
print(f'Entrada: {linea_error}')
print(f'Resultado: {parse_error_log(linea_error)}')

linea_auth = '[AUTH] 2024-03-15 10:30:00 | user=admin@empresa.com | action=LOGIN | status=SUCCESS | ip=10.0.0.5 | session=abc123xyz'
print('\n-- Parser Auth --')
print(f'Entrada: {linea_auth}')
print(f'Resultado: {parse_auth_log(linea_auth)}')

linea_db = "[DB-2024-03-15 10:35:22] QUERY executed in 0.045s: SELECT * FROM users WHERE email = 'admin@empresa.com'"
print('\n-- Parser DB --')
print(f'Entrada: {linea_db}')
print(f'Resultado: {parse_db_log(linea_db)}')

PRUEBA DE PARSERS

-- Parser HTTP --
Entrada: 192.168.1.100 - - [15/Mar/2024:10:23:45 -0600] "GET /api/users...
Resultado: {'ip': '192.168.1.100', 'timestamp': '15/Mar/2024:10:23:45 -0600', 'method': 'GET', 'path': '/api/users', 'status': 200, 'bytes': 1234, 'referer': 'https://ejemplo.com', 'user_agent': 'Mozilla/5.0'}

-- Parser Error --
Entrada: [2024-03-15 10:25:12] ERROR app.database - DatabaseConnectionError: Connection refused
Resultado: {'timestamp': '2024-03-15 10:25:12', 'level': 'ERROR', 'module': 'app.database', 'error_type': 'DatabaseConnectionError', 'message': 'Connection refused'}

-- Parser Auth --
Entrada: [AUTH] 2024-03-15 10:30:00 | user=admin@empresa.com | action=LOGIN | status=SUCCESS | ip=10.0.0.5 | session=abc123xyz
Resultado: {'timestamp': '2024-03-15 10:30:00', 'user': 'admin@empresa.com', 'action': 'LOGIN', 'status': 'SUCCESS', 'ip': '10.0.0.5', 'extra': {'session': 'abc123xyz'}}

-- Parser DB --
Entrada: [DB-2024-03-15 10:35:22] QUERY executed in 0.045s: SEL

In [ ]:
print('\nGENERANDO REPORTE COMPLETO...\n')
reporte = generar_reporte(LOGS_PRUEBA)
mostrar_reporte(reporte)


GENERANDO REPORTE COMPLETO...

                    REPORTE DE ANALISIS DE LOGS

 Resumen GENERAL
----------------------------------------
Total de lineas procesadas: 24
Por tipo:
  - HTTP: 7
  - ERROR: 5
  - AUTH: 6
  - DB: 6

 Logs HTTP
----------------------------------------
Total requests: 7
Por codigo de estado:
  - 2xx: 4
  - 3xx: 0
  - 4xx: 2
  - 5xx: 1
Top 5 rutas mas solicitadas:
  - /api/users: 3 requests
  - /api/login: 1 requests
  - /admin/../../../etc/passwd: 1 requests
  - /products?id=1: 1 requests
  - /products?id=2: 1 requests

 Errores
----------------------------------------
Total errores: 4
Por nivel:
  - ERROR: 3
  - WARNING: 1

 Alertas de Seguridad
----------------------------------------
Posibles ataques de fuerza bruta: 1
  IP: 192.168.1.50 - 4 intentos fallidos
Posibles SQL Injection: 2
  Query: SELECT * FROM users WHERE username = 'admin' OR 1=1--'...
  Query: SELECT * FROM users UNION SELECT * FROM passwords...
Posibles Path Traversal: 1
  Ruta: /admin/../

In [ ]:
# ── BONUS ────────────────────────────────────────────────────────────────

def exportar_reporte_json(reporte: Dict, archivo: str) -> None:
    with open(archivo, 'w', encoding='utf-8') as f:
        _json.dump(reporte, f, ensure_ascii=False, indent=2, default=str)
    print(f'Reporte exportado a: {archivo}')


def analisis_temporal(logs_http: List[Dict]) -> Dict:
    horas: Counter = Counter()
    for h in logs_http:
        m = re.search(r':(\d{2}):', h.get('timestamp', ''))
        if m:
            horas[int(m.group(1))] += 1
    return dict(sorted(horas.items()))


BOT_USER_AGENTS = re.compile(
    r'(?i)\b(curl|wget|python-requests|scrapy|bot|spider|crawler|'
    r'sqlmap|nikto|nmap|masscan|libwww-perl|PostmanRuntime|Go-http-client)\b'
)

def detectar_bots(logs_http: List[Dict]) -> List[Dict]:
    return [h for h in logs_http if BOT_USER_AGENTS.search(h.get('user_agent', ''))]